In [2]:
import pandas as pd
import re
import html
import time
import json
import requests
import pandas as pd
from datetime import datetime, timezone
from tqdm import tqdm

In [4]:


ALGOLIA_URL = "https://hn.algolia.com/api/v1/search_by_date"

def fetch_hn(keyword, tag="story", hits_per_page=100, max_pages=2):
    rows = []
    for page in range(max_pages):
        params = {
            "query": keyword,
            "tags": tag,            # IMPORTANT: ONLY ONE TAG here
            "hitsPerPage": hits_per_page,
            "page": page
        }
        r = requests.get(ALGOLIA_URL, params=params, timeout=30)
        r.raise_for_status()
        data = r.json()

        for hit in data.get("hits", []):
            rows.append({
                "source": "HackerNews",
                "keyword": keyword,
                "content_type": tag,
                "objectID": hit.get("objectID"),
                "created_at": hit.get("created_at"),
                "author": hit.get("author"),
                "url": hit.get("url"),
                "story_title": hit.get("title") or hit.get("story_title"),
                "text": hit.get("comment_text") or hit.get("story_text"),
            })
    return rows

In [5]:
KEYWORDS_TO_TRACK = ["film", "platform", "creator", "monetization"]

all_rows = []

for kw in tqdm(KEYWORDS_TO_TRACK):
    story_rows = fetch_hn(kw, tag="story", max_pages=3)
    comment_rows = fetch_hn(kw, tag="comment", max_pages=3)

    print(f"{kw}: stories={len(story_rows)} comments={len(comment_rows)}")

    all_rows.extend(story_rows)
    all_rows.extend(comment_rows)

df_raw = pd.DataFrame(all_rows).drop_duplicates(subset=["objectID"])
df_raw["fetched_at"] = datetime.now(timezone.utc).isoformat()

df_raw.shape, df_raw.columns

 25%|████████████████████████████████████████████▌                                                                                                                                     | 1/4 [00:03<00:11,  3.77s/it]

film: stories=300 comments=300


 50%|█████████████████████████████████████████████████████████████████████████████████████████                                                                                         | 2/4 [00:07<00:07,  3.79s/it]

platform: stories=300 comments=300


 75%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████▌                                            | 3/4 [00:11<00:03,  3.80s/it]

creator: stories=300 comments=300


100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 4/4 [00:15<00:00,  3.77s/it]

monetization: stories=300 comments=300


((2267, 10),
 Index(['source', 'keyword', 'content_type', 'objectID', 'created_at', 'author',
        'url', 'story_title', 'text', 'fetched_at'],
       dtype='object'))

In [6]:
import re
import html

def clean_text_fn(s: str) -> str:
    if s is None:
        return ""
    s = str(s)

    # Decode HTML entities: &gt; &quot; &#x2F; etc.
    s = html.unescape(s)

    # Remove HTML tags like <p> <pre> <code> <a href=...>
    s = re.sub(r"<[^>]+>", " ", s)

    # Remove leftover escaped artifacts
    s = s.replace("\\n", " ").replace("\\t", " ")

    # Collapse whitespace
    s = re.sub(r"\s+", " ", s).strip()

    return s

In [7]:
df = df_raw.copy()

df["clean_text"] = df["text"].apply(clean_text_fn)
df["clean_length"] = df["clean_text"].str.len()

# Drop empty and very short texts (tune threshold if needed)
df = df[df["clean_length"] >= 40].copy()

df.shape, df[["text", "clean_text"]].head(3)

((1880, 12),
                                                 text  \
 1  I wanted to ship a clean, polished video proce...   
 4  Hi, author of the repo speaking here!<p>When I...   
 5  We’ve all seen the viral AI video clips: stunn...   
 
                                           clean_text  
 1  I wanted to ship a clean, polished video proce...  
 4  Hi, author of the repo speaking here! When I t...  
 5  We’ve all seen the viral AI video clips: stunn...  )

In [8]:
LUNIM_KEYWORDS = [
    "market gap", "customer need", "user need",
    "pain point", "problem", "friction",
    "platform", "tool", "product", "solution", "alternative",
    "use case", "value", "benefit", "advantage",
    "adoption", "workflow", "efficiency", "automation",
    "pricing", "cost", "revenue", "monetization"
]

NOISE_KEYWORDS = [
    # force job stuff to False
    "who is hiring", "hiring", "job", "role", "position", "resume", "cv",
    # common HN noise
    "kernel", "compiler", "filesystem", "mmap", "rust", "c++", "linux",
    "crypto", "token", "blockchain"
]

def mark_lunim_relevance(text: str) -> bool:
    if not isinstance(text, str):
        return False
    t = text.lower()

    # hard reject noise first
    if any(nk in t for nk in NOISE_KEYWORDS):
        return False

    # accept if any market keyword appears
    return any(lk in t for lk in LUNIM_KEYWORDS)

In [10]:
df["is_relevant"] = df["clean_text"].apply(mark_lunim_relevance)

print(df["is_relevant"].value_counts())
df[df["is_relevant"] == True][["clean_text"]].head(5)

is_relevant
True     1003
False     877
Name: count, dtype: int64


,clean_text
1,"I wanted to ship a clean, polished video proce..."
4,"Hi, author of the repo speaking here! When I t..."
12,"Hi HN — I’m the developer of NetViews, a macOS..."
13,What this is: Copy-paste LLM prompts that turn...
20,"Hi HN, I built SpecOps, an open-source CLI fra..."


In [11]:
df_relevant = df[df["is_relevant"] == True].copy()
df_noise    = df[df["is_relevant"] == False].copy()

df_relevant.shape, df_noise.shape

((1003, 13), (877, 13))

In [12]:
from datetime import datetime

ts = datetime.now().strftime("%Y%m%d_%H%M")
df.to_csv(f"hn_market_labeled_{ts}.csv", index=False)
df_relevant.to_csv(f"hn_market_relevant_{ts}.csv", index=False)
df_noise.to_csv(f"hn_market_noise_{ts}.csv", index=False)

print("Saved files with timestamp:", ts)

Saved files with timestamp: 20260210_0954


In [13]:
# Existing timestamp saves (keep)
df.to_csv(f"hn_market_labeled_{ts}.csv", index=False)
df_relevant.to_csv(f"hn_market_relevant_{ts}.csv", index=False)
df_noise.to_csv(f"hn_market_noise_{ts}.csv", index=False)

# ✅ Add these "latest" outputs (new)
df.to_csv("hn_market_labeled_latest.csv", index=False)
df_relevant.to_csv("hn_market_relevant_latest.csv", index=False)
df_noise.to_csv("hn_market_noise_latest.csv", index=False)